# Day 14 - Logging, config, outputs and incremental runs

All tasks are in this notebook. Use **Run All** from the top, then run the last code cell again to check task 10.

## Setup

In [ ]:
import csv
import hashlib
import json
import logging
import math
import re
import time
from collections import Counter
from datetime import datetime
from pathlib import Path

import yaml


## Task 1: logging instead of print
Every message goes through a logger at one of three levels: INFO (progress), WARNING (something to check), ERROR (something failed).

In [ ]:
logger = logging.getLogger("pipeline")
logger.setLevel(logging.INFO)
logger.propagate = False


def set_handlers(log_file: Path | None = None) -> None:
    """Replace the handlers: always the console, plus a file when log_file is given."""
    for handler in list(logger.handlers):
        handler.close()
        logger.removeHandler(handler)
    formatter = logging.Formatter("%(asctime)s %(levelname)-8s %(message)s")
    handlers = [logging.StreamHandler()]
    if log_file is not None:
        handlers.append(logging.FileHandler(log_file, encoding="utf-8"))
    for handler in handlers:
        handler.setFormatter(formatter)
        logger.addHandler(handler)


set_handlers()
logger.info("INFO: normal progress, such as a file being processed")
logger.warning("WARNING: something to check, such as a rejected row")
logger.error("ERROR: something failed, such as a file that cannot be read")


## Task 2: a log file per run, with a timestamp
Each run writes `logs/run_<timestamp>.log`, so earlier logs are kept.

In [ ]:
def new_run_id() -> str:
    """A timestamp used in the names of this run's log, summary and bad records files."""
    return datetime.now().strftime("%Y%m%d_%H%M%S")


def start_run_log(log_dir: str, run_id: str) -> Path:
    """Create logs/run_<run_id>.log and send all later messages to it as well."""
    Path(log_dir).mkdir(parents=True, exist_ok=True)
    log_file = Path(log_dir) / f"run_{run_id}.log"
    set_handlers(log_file)
    return log_file


logger.info("Example run id: %s", new_run_id())


## Task 3: paths, thresholds and mappings come from config.yaml
The settings are read once here. Nothing below has a path or threshold typed into it.

In [ ]:
CONFIG_FILE = "config.yaml"

REQUIRED_CONFIG_KEYS = [
    "input_dir", "output_dir", "log_dir", "manifest_file", "max_reject_rate",
    "date_format", "zip_pattern", "id_pattern", "allowed_statuses", "files",
    "column_mapping", "float_columns", "integer_columns", "date_columns",
]

with open(CONFIG_FILE, encoding="utf-8") as f:
    CONFIG = yaml.safe_load(f)

missing = [key for key in REQUIRED_CONFIG_KEYS if key not in CONFIG]
if missing:
    raise ValueError(f"config.yaml is missing settings: {missing}")

logger.info("Loaded %s: input=%s, output=%s, limit=%s",
            CONFIG_FILE, CONFIG["input_dir"], CONFIG["output_dir"], CONFIG["max_reject_rate"])


## Validation rules (day 13 checks, now read from config.yaml)
This section is the validation from day 13. It is needed by tasks 4 to 7.

In [ ]:
class SchemaError(Exception):
    """A file is missing required columns."""


class FileFormatError(Exception):
    """A file is empty, has no header row, or is not valid UTF-8."""


INPUT_DIR = Path(CONFIG["input_dir"])
OUTPUT_DIR = Path(CONFIG["output_dir"])
MANIFEST_FILE = Path(CONFIG["manifest_file"])
MAX_REJECT_RATE = float(CONFIG["max_reject_rate"])
DATE_FORMAT = CONFIG["date_format"]
ZIP_PATTERN = re.compile(CONFIG["zip_pattern"])
ID_PATTERN = re.compile(CONFIG["id_pattern"])
ALLOWED_STATUSES = frozenset(CONFIG["allowed_statuses"])
REQUIRED = {name: spec["required"] for name, spec in CONFIG["files"].items()}
COLUMN_MAPPING = CONFIG["column_mapping"]
FLOAT_COLUMNS = frozenset(CONFIG["float_columns"])
INTEGER_COLUMNS = frozenset(CONFIG["integer_columns"])
DATE_COLUMNS = frozenset(CONFIG["date_columns"])

# Changes when any rule in config.yaml changes, so affected files are checked again
RULE_KEYS = [
    "files", "max_reject_rate", "date_format", "zip_pattern", "id_pattern",
    "allowed_statuses", "column_mapping", "float_columns", "integer_columns",
    "date_columns",
]
RULES_FINGERPRINT = hashlib.sha256(
    json.dumps({key: CONFIG[key] for key in RULE_KEYS}, sort_keys=True).encode("utf-8")
).hexdigest()


def _clean(value) -> str:
    """Return the value without surrounding spaces, or '' if it is missing."""
    return (value or "").strip()


def check_id(value: str) -> str | None:
    """IDs must be exactly 32 hexadecimal characters."""
    return None if ID_PATTERN.fullmatch(value) else "must be 32 hex characters"


def check_zip(value: str) -> str | None:
    """Zip prefix must be exactly 5 digits."""
    return None if ZIP_PATTERN.fullmatch(value) else "zip prefix must be 5 digits"


def check_status(value: str) -> str | None:
    """Order status must be on the allowed list."""
    return None if value in ALLOWED_STATUSES else f"status '{value}' is not allowed"


def check_timestamp(value: str) -> str | None:
    """Value must parse as a date in the configured format."""
    try:
        datetime.strptime(value, DATE_FORMAT)
    except ValueError:
        return f"'{value}' is not a date in the format {DATE_FORMAT}"
    return None


def check_number(value: str) -> str | None:
    """Value must be a finite number."""
    try:
        number = float(value)
    except ValueError:
        return f"'{value}' is not a number"
    return None if math.isfinite(number) else f"'{value}' is not a finite number"


def check_price(value: str) -> str | None:
    """Price must be greater than 0."""
    try:
        number = float(value)
    except ValueError:
        return f"'{value}' is not a number"
    return None if number > 0 else "price must be greater than 0"


def check_freight(value: str) -> str | None:
    """Freight must be 0 or more."""
    try:
        number = float(value)
    except ValueError:
        return f"'{value}' is not a number"
    return None if number >= 0 else "freight must be 0 or more"


# Checks for each column, run in order. A column stops at its first failed check.
COLUMN_CHECKS = {
    "order_id": [check_id],
    "customer_id": [check_id],
    "product_id": [check_id],
    "seller_id": [check_id],
    "customer_zip_code_prefix": [check_zip],
    "order_status": [check_status],
    "order_purchase_timestamp": [check_timestamp],
    "order_delivered_customer_date": [check_timestamp],
    "price": [check_number, check_price],
    "freight_value": [check_number, check_freight],
}


def validate_row(row: dict, required: list[str]) -> list[str]:
    """Return every problem in one row. An empty list means the row is valid."""
    errors = []
    for col in required:
        if _clean(row.get(col)) == "":
            errors.append(f"{col} is blank")
    for col, checks in COLUMN_CHECKS.items():
        value = _clean(row.get(col))
        if value == "":
            continue
        for check in checks:
            problem = check(value)
            if problem:
                errors.append(f"{col}: {problem}")
                break
    return errors


def read_rows(path: Path) -> tuple[list[str], list[dict]]:
    """Read a CSV file and return (header, rows)."""
    try:
        with open(path, newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            header = reader.fieldnames
            rows = list(reader)
    except UnicodeDecodeError as err:
        raise FileFormatError(f"{path.name} is not valid UTF-8: {err}") from err
    if not header:
        raise FileFormatError(f"{path.name} is empty or has no header row")
    return header, rows


def check_schema(filename: str, header: list[str], required: list[str]) -> None:
    """Raise SchemaError if a required column is missing."""
    missing = [col for col in required if col not in header]
    if missing:
        raise SchemaError(f"{filename} is missing required columns: {missing}")


def normalise_row(row: dict, header: list[str]) -> dict:
    """Rename columns to the standard schema and convert numbers and dates."""
    clean = {}
    for col in header:
        name = COLUMN_MAPPING.get(col, col)
        value = _clean(row.get(col))
        if value == "":
            clean[name] = None
        elif name in FLOAT_COLUMNS:
            clean[name] = float(value)
        elif name in INTEGER_COLUMNS:
            clean[name] = int(value)
        elif name in DATE_COLUMNS:
            clean[name] = datetime.strptime(value, DATE_FORMAT).isoformat(sep=" ")
        else:
            clean[name] = value
    return clean


def validate_file(path: Path, filename: str) -> dict:
    """Check the columns and every row, collect the rejects, and apply the limit."""
    header, rows = read_rows(path)
    required = REQUIRED[filename]
    check_schema(filename, header, required)

    result = {
        "file": filename,
        "status": "OK",
        "message": "",
        "total": len(rows),
        "clean_columns": [COLUMN_MAPPING.get(col, col) for col in header],
        "valid": [],
        "rejected": [],
    }
    for line_no, row in enumerate(rows, start=2):  # line 1 is the header
        errors = validate_row(row, required)
        if errors:
            record = "; ".join(f"{k}={_clean(v)}" for k, v in row.items() if k is not None)
            result["rejected"].append({
                "file": filename,
                "line": line_no,
                "reason": "; ".join(errors),
                "record": record,
            })
        else:
            result["valid"].append(normalise_row(row, header))

    rate = len(result["rejected"]) / len(rows) if rows else 0.0
    if rate > MAX_REJECT_RATE:
        result["status"] = "FAILED"
        result["message"] = f"{rate:.0%} of rows rejected, above the {MAX_REJECT_RATE:.0%} limit"
    return result


def write_bad_records(rejected: list[dict], path: Path) -> None:
    """Write every rejected row, with its reason, to one CSV file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["file", "line", "reason", "record"])
        writer.writeheader()
        writer.writerows(rejected)


def summary_row(filename: str, status: str, message: str = "", total: int = 0,
                valid: int = 0, rejected: int = 0) -> dict:
    """One line of the run summary for one file."""
    return {"file": filename, "status": status, "message": message,
            "total": total, "valid": valid, "rejected": rejected}


## Task 4: write the clean output as CSV and JSON
Only valid rows are written. Column names follow the standard schema, prices are numbers, and dates are in one format.

In [ ]:
def write_clean_output(result: dict, output_dir: Path) -> tuple[Path, Path]:
    """Write the valid rows as a CSV file and as a JSON file. Return both paths."""
    output_dir.mkdir(parents=True, exist_ok=True)
    stem = Path(result["file"]).stem
    csv_path = output_dir / f"{stem}_clean.csv"
    json_path = output_dir / f"{stem}_clean.json"

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=result["clean_columns"])
        writer.writeheader()
        writer.writerows(result["valid"])

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(result["valid"], f, indent=2, ensure_ascii=False)
    return csv_path, json_path


## Task 5: run summary as JSON (counts and duration)

In [ ]:
def build_summary(run_id: str, started_at: datetime, duration: float,
                  log_file: Path, file_summaries: list[dict], bad_path: Path) -> dict:
    """Combine the per-file results into one summary for the run."""
    counts = Counter(s["status"] for s in file_summaries)
    validated = [s for s in file_summaries if s["status"] in ("OK", "FAILED")]
    return {
        "run_id": run_id,
        "started_at": started_at.isoformat(timespec="seconds"),
        "duration_seconds": round(duration, 3),
        "log_file": str(log_file),
        "files_validated": len(validated),
        "files_ok": counts["OK"],
        "files_failed": counts["FAILED"],
        "files_skipped": counts["SKIPPED"],
        "files_missing": counts["MISSING"],
        "rows_validated": sum(s["total"] for s in validated),
        "rows_valid": sum(s["valid"] for s in validated),
        "rows_rejected": sum(s["rejected"] for s in validated),
        "bad_records_file": str(bad_path),
        "files": file_summaries,
    }


def write_run_summary(summary: dict, output_dir: Path) -> Path:
    """Save the run summary as JSON, named by run id."""
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / f"run_summary_{summary['run_id']}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)
    return path


## Task 6: manifest of processed files, so unchanged files are skipped
Each file is fingerprinted with SHA-256. A file is skipped when its fingerprint and the rules are unchanged since the last run.

In [ ]:
def file_sha256(path: Path) -> str:
    """Return the SHA-256 fingerprint of a file's contents."""
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            digest.update(chunk)
    return digest.hexdigest()


def load_manifest() -> dict:
    """Read the record of processed files, or start empty on the first run."""
    if not MANIFEST_FILE.exists():
        return {}
    with open(MANIFEST_FILE, encoding="utf-8") as f:
        return json.load(f)


def save_manifest(manifest: dict) -> None:
    """Save the record of processed files."""
    MANIFEST_FILE.parent.mkdir(parents=True, exist_ok=True)
    with open(MANIFEST_FILE, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)


def is_unchanged(manifest: dict, filename: str, digest: str) -> bool:
    """True when the file and the rules match the last run."""
    entry = manifest.get(filename)
    return bool(entry) and entry["sha256"] == digest and entry["rules"] == RULES_FINGERPRINT


def manifest_entry(digest: str, status: str, total: int, valid: int, rejected: int) -> dict:
    """The record kept for one file after it has been processed."""
    return {
        "sha256": digest,
        "rules": RULES_FINGERPRINT,
        "status": status,
        "total": total,
        "valid": valid,
        "rejected": rejected,
        "processed_at": datetime.now().isoformat(timespec="seconds"),
    }


## Task 7: the whole pipeline from one cell
This cell defines the pipeline and runs it. It uses the settings from config.yaml.

In [ ]:
def run_pipeline() -> dict:
    """Validate every file, write the outputs, update the manifest, and return the summary."""
    started_at = datetime.now()
    timer = time.perf_counter()
    run_id = new_run_id()
    log_file = start_run_log(CONFIG["log_dir"], run_id)
    logger.info("Run %s started with %s", run_id, CONFIG_FILE)
    logger.info("Log file: %s", log_file)

    manifest = load_manifest()
    file_summaries = []
    all_rejected = []

    for filename in REQUIRED:
        path = INPUT_DIR / filename
        if not path.exists():
            logger.warning("%s not found in %s, skipped", filename, INPUT_DIR)
            file_summaries.append(summary_row(filename, "MISSING", message=f"not found in {INPUT_DIR}"))
            continue

        digest = file_sha256(path)
        if is_unchanged(manifest, filename, digest):
            previous = manifest[filename]
            logger.info("%s unchanged since last run (status %s), skipped",
                        filename, previous["status"])
            file_summaries.append(summary_row(
                filename, "SKIPPED",
                message=f"unchanged since last run, status {previous['status']}",
                total=previous["total"], valid=previous["valid"], rejected=previous["rejected"],
            ))
            continue

        try:
            result = validate_file(path, filename)
        except (SchemaError, FileFormatError) as err:
            logger.error("%s failed: %s", filename, err)
            file_summaries.append(summary_row(filename, "FAILED", message=str(err)))
            manifest[filename] = manifest_entry(digest, "FAILED", 0, 0, 0)
            continue

        if result["status"] == "OK":
            csv_path, json_path = write_clean_output(result, OUTPUT_DIR)
            logger.info("%s processed: %d valid, %d rejected of %d rows",
                        filename, len(result["valid"]), len(result["rejected"]), result["total"])
            logger.info("Clean output written: %s and %s", csv_path, json_path)
        else:
            logger.error("%s failed: %s", filename, result["message"])

        if result["rejected"]:
            logger.warning("%s: %d rows rejected, see the bad records file",
                           filename, len(result["rejected"]))
        all_rejected.extend(result["rejected"])

        manifest[filename] = manifest_entry(
            digest, result["status"], result["total"],
            len(result["valid"]), len(result["rejected"]),
        )
        file_summaries.append(summary_row(
            filename, result["status"], message=result["message"], total=result["total"],
            valid=len(result["valid"]), rejected=len(result["rejected"]),
        ))

    bad_path = OUTPUT_DIR / f"bad_records_{run_id}.csv"
    write_bad_records(all_rejected, bad_path)
    save_manifest(manifest)

    summary = build_summary(run_id, started_at, time.perf_counter() - timer,
                            log_file, file_summaries, bad_path)
    summary_path = write_run_summary(summary, OUTPUT_DIR)
    logger.info("Run %s finished in %.2fs: %d validated, %d skipped, %d failed, %d missing",
                run_id, summary["duration_seconds"], summary["files_validated"],
                summary["files_skipped"], summary["files_failed"], summary["files_missing"])
    logger.info("Run summary written: %s", summary_path)
    return summary


summary = run_pipeline()
summary


## Task 8: lint with ruff
No code cell. In the terminal, from the project folder, run:

```
ruff check day14.ipynb
```

Ruff checks the code in the notebook cells. Fix any warnings in the cells, then run it again until it reports `All checks passed!`.

## Task 9: review and merge
No code cell. Commit `day14.ipynb` and `config.yaml` on a branch, open a pull request, address the reviewer's comments with new commits, then merge.

## Task 10: second run processes nothing new
Run the **Task 7** cell a second time. In the new summary, `files_validated` should be **0** and `files_skipped` should be **3**. Compare it with the first run's summary.